# MMF Data Preparation — Reproducibility Notebook
Auto-generated by mmf-agent after interactive `/prep-and-clean-data` session.

This notebook records all decisions made during the interactive data preparation
step and can be re-run to reproduce the exact same training table from the source data.

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
source_table = "{source_table}"
unique_id_col = "{unique_id_col}"
ds_col = "{ds_col}"
y_col = "{y_col}"
freq = "{freq}"
imputation_method = "{imputation_method}"
exclusion_threshold = {exclusion_threshold}
iqr_multiplier = {iqr_multiplier}

In [ ]:
source_fqn = f"{catalog}.{schema}.{source_table}"
train_table = f"{catalog}.{schema}.{use_case}_train_data"

print(f"Source: {source_fqn}")
row_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {source_fqn}").collect()[0]["cnt"]
date_range = spark.sql(f"SELECT MIN({ds_col}) AS min_ds, MAX({ds_col}) AS max_ds FROM {source_fqn}").collect()[0]
n_series = spark.sql(f"SELECT COUNT(DISTINCT {unique_id_col}) AS n FROM {source_fqn}").collect()[0]["n"]
print(f"  Rows: {row_count:,}")
print(f"  Date range: {date_range['min_ds']} to {date_range['max_ds']}")
print(f"  Series: {n_series:,}")
print(f"  Frequency: {freq}")
print(f"  Mapping: unique_id={unique_id_col}, ds={ds_col}, y={y_col}")

In [ ]:
source_fqn = f"{catalog}.{schema}.{source_table}"
train_table = f"{catalog}.{schema}.{use_case}_train_data"

if freq == "H":
    sql = f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT CAST({unique_id_col} AS STRING) AS unique_id,
           CAST({ds_col} AS TIMESTAMP) AS ds,
           CAST({y_col} AS DOUBLE) AS y
    FROM {source_fqn} WHERE {y_col} IS NOT NULL"""
elif freq == "D":
    sql = f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT CAST({unique_id_col} AS STRING) AS unique_id,
           CAST({ds_col} AS DATE) AS ds,
           CAST({y_col} AS DOUBLE) AS y
    FROM {source_fqn} WHERE {y_col} IS NOT NULL"""
elif freq == "W":
    sql = f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT CAST({unique_id_col} AS STRING) AS unique_id,
           CAST(DATE_TRUNC('week', {ds_col}) + INTERVAL 6 DAY AS DATE) AS ds,
           SUM(CAST({y_col} AS DOUBLE)) AS y
    FROM {source_fqn} WHERE {y_col} IS NOT NULL
    GROUP BY {unique_id_col}, DATE_TRUNC('week', {ds_col}) + INTERVAL 6 DAY"""
elif freq == "M":
    sql = f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT CAST({unique_id_col} AS STRING) AS unique_id,
           CAST(LAST_DAY({ds_col}) AS DATE) AS ds,
           SUM(CAST({y_col} AS DOUBLE)) AS y
    FROM {source_fqn} WHERE {y_col} IS NOT NULL
    GROUP BY {unique_id_col}, LAST_DAY({ds_col})"""

spark.sql(sql)
count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {train_table}").collect()[0]["cnt"]
print(f"Created {train_table}: {count:,} rows")
spark.sql(f"SELECT * FROM {train_table} ORDER BY unique_id, ds LIMIT 5").show()

In [ ]:
interval_map = {"H": "INTERVAL 1 HOUR", "D": "INTERVAL 1 DAY", "W": "INTERVAL 7 DAY", "M": "INTERVAL 1 MONTH"}
interval = interval_map[freq]
train_table = f"{catalog}.{schema}.{use_case}_train_data"

missing_df = spark.sql(f"""
WITH global_max AS (
  SELECT MAX(ds) AS max_ds FROM {train_table}
),
series_bounds AS (
  SELECT unique_id, MIN(ds) AS min_ds
  FROM {train_table} GROUP BY unique_id
),
date_spine AS (
  SELECT s.unique_id, EXPLODE(SEQUENCE(s.min_ds, g.max_ds, {interval})) AS expected_ds
  FROM series_bounds s CROSS JOIN global_max g
),
spine_joined AS (
  SELECT sp.unique_id, sp.expected_ds AS ds, t.y
  FROM date_spine sp
  LEFT JOIN {train_table} t ON sp.unique_id = t.unique_id AND sp.expected_ds = t.ds
)
SELECT unique_id,
  COUNT(*) AS expected_count,
  SUM(CASE WHEN y IS NOT NULL THEN 1 ELSE 0 END) AS actual_count,
  SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) AS missing_count,
  ROUND(SUM(CASE WHEN y IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS missing_pct
FROM spine_joined GROUP BY unique_id ORDER BY missing_pct DESC
""")
missing_df.createOrReplaceTempView("missing_assessment")

total = missing_df.count()
n_clean = missing_df.filter("missing_count = 0").count()
n_low = missing_df.filter("missing_pct > 0 AND missing_pct < 5").count()
n_mid = missing_df.filter("missing_pct >= 5 AND missing_pct <= 20").count()
n_high = missing_df.filter(f"missing_pct > {exclusion_threshold}").count()

trailing_df = spark.sql(f"""
WITH global_max AS (
  SELECT MAX(ds) AS max_ds FROM {train_table}
),
series_last AS (
  SELECT unique_id, MAX(ds) AS last_ds
  FROM {train_table} WHERE y IS NOT NULL GROUP BY unique_id
)
SELECT s.unique_id, s.last_ds, g.max_ds,
       DATEDIFF(g.max_ds, s.last_ds) AS trailing_gap_days
FROM series_last s CROSS JOIN global_max g
WHERE s.last_ds < g.max_ds
ORDER BY trailing_gap_days DESC
""")
n_trailing = trailing_df.count()

print("Missing data summary:")
print(f"  Complete (no gaps): {n_clean}")
print(f"  Low (< 5%):         {n_low}")
print(f"  Medium (5-20%):     {n_mid}")
print(f"  High (> {exclusion_threshold}%):      {n_high}")
if n_trailing > 0:
    global_max_ds = trailing_df.select("max_ds").first()[0]
    print(f"\nTrailing gaps: {n_trailing} series end before global max ({global_max_ds})")
    trailing_df.show(50, truncate=False)
print(f"\nImputation: {imputation_method}")
print(f"Exclusion threshold: {exclusion_threshold}%")

In [ ]:
interval_map = {"H": "INTERVAL 1 HOUR", "D": "INTERVAL 1 DAY", "W": "INTERVAL 7 DAY", "M": "INTERVAL 1 MONTH"}
interval = interval_map[freq]
train_table = f"{catalog}.{schema}.{use_case}_train_data"

excluded_series = []
if exclusion_threshold > 0:
    excluded_rows = spark.sql(f"""
    SELECT unique_id FROM missing_assessment WHERE missing_pct > {exclusion_threshold}
    """).collect()
    excluded_series = [r["unique_id"] for r in excluded_rows]

spark.sql(f"""
CREATE OR REPLACE TABLE {train_table} AS
WITH global_max AS (
  SELECT MAX(ds) AS max_ds FROM {train_table}
),
series_bounds AS (
  SELECT unique_id, MIN(ds) AS min_ds
  FROM {train_table} GROUP BY unique_id
),
date_spine AS (
  SELECT s.unique_id, EXPLODE(SEQUENCE(s.min_ds, g.max_ds, {interval})) AS expected_ds
  FROM series_bounds s CROSS JOIN global_max g
)
SELECT sp.unique_id, sp.expected_ds AS ds, t.y
FROM date_spine sp
LEFT JOIN {train_table} t ON sp.unique_id = t.unique_id AND sp.expected_ds = t.ds
""")

if excluded_series:
    placeholders = ",".join([f"\'{ s }\'" for s in excluded_series])
    spark.sql(f"DELETE FROM {train_table} WHERE unique_id IN ({placeholders})")
    print(f"Excluded {len(excluded_series)} series with > {exclusion_threshold}% missing data")

pre_null_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {train_table} WHERE y IS NULL").collect()[0]["cnt"]

if imputation_method == "interpolation":
    spark.sql(f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT unique_id, ds,
      COALESCE(y, (
        LAG(y IGNORE NULLS) OVER (PARTITION BY unique_id ORDER BY ds) +
        LEAD(y IGNORE NULLS) OVER (PARTITION BY unique_id ORDER BY ds)
      ) / 2) AS y
    FROM {train_table}
    """)
elif imputation_method == "forward_fill":
    spark.sql(f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT unique_id, ds,
      COALESCE(y, LAST_VALUE(y IGNORE NULLS) OVER (PARTITION BY unique_id ORDER BY ds)) AS y
    FROM {train_table}
    """)
elif imputation_method == "fill_zero":
    spark.sql(f"UPDATE {train_table} SET y = 0 WHERE y IS NULL")

post_null_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {train_table} WHERE y IS NULL").collect()[0]["cnt"]
total_rows = spark.sql(f"SELECT COUNT(*) AS cnt FROM {train_table}").collect()[0]["cnt"]
print(f"Imputation complete: {pre_null_count - post_null_count:,} values filled")
print(f"  Rows: {total_rows:,}, remaining nulls: {post_null_count:,}")

In [ ]:
train_table = f"{catalog}.{schema}.{use_case}_train_data"

anomaly_df = spark.sql(f"""
WITH stats AS (
  SELECT unique_id,
         PERCENTILE(y, 0.25) AS q1, PERCENTILE(y, 0.75) AS q3,
         PERCENTILE(y, 0.75) - PERCENTILE(y, 0.25) AS iqr
  FROM {train_table} GROUP BY unique_id
),
flagged AS (
  SELECT t.unique_id, t.ds, t.y, s.q1, s.q3, s.iqr,
         CASE WHEN t.y < s.q1 - 1.5 * s.iqr OR t.y > s.q3 + 1.5 * s.iqr
              THEN TRUE ELSE FALSE END AS is_anomaly
  FROM {train_table} t JOIN stats s ON t.unique_id = s.unique_id
)
SELECT unique_id, COUNT(*) AS total_points,
       SUM(CASE WHEN is_anomaly THEN 1 ELSE 0 END) AS anomaly_count,
       ROUND(SUM(CASE WHEN is_anomaly THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS anomaly_pct,
       MIN(CASE WHEN is_anomaly THEN y END) AS min_anomaly,
       MAX(CASE WHEN is_anomaly THEN y END) AS max_anomaly
FROM flagged GROUP BY unique_id HAVING anomaly_count > 0 ORDER BY anomaly_pct DESC
""")

n_affected = anomaly_df.count()
total_series = spark.sql(f"SELECT COUNT(DISTINCT unique_id) AS n FROM {train_table}").collect()[0]["n"]
total_anomalies = anomaly_df.agg({"anomaly_count": "sum"}).collect()[0][0] or 0
print(f"Anomaly detection (1.5x IQR):")
print(f"  Series with anomalies: {n_affected} / {total_series}")
print(f"  Total anomalous points: {int(total_anomalies):,}")
print(f"\nCapping: {iqr_multiplier}x IQR" + (" (skipped)" if iqr_multiplier == 0 else ""))
if n_affected > 0:
    print("\nTop affected series:")
    anomaly_df.show(10, truncate=False)

In [ ]:
train_table = f"{catalog}.{schema}.{use_case}_train_data"
capped_count = 0

if iqr_multiplier > 0:
    cap_detail = spark.sql(f"""
    SELECT t.unique_id,
           SUM(CASE WHEN t.y < s.q1 - {iqr_multiplier} * s.iqr
                         OR t.y > s.q3 + {iqr_multiplier} * s.iqr
                    THEN 1 ELSE 0 END) AS capped_count
    FROM {train_table} t
    JOIN (
      SELECT unique_id, PERCENTILE(y, 0.25) AS q1, PERCENTILE(y, 0.75) AS q3,
             PERCENTILE(y, 0.75) - PERCENTILE(y, 0.25) AS iqr
      FROM {train_table} GROUP BY unique_id
    ) s ON t.unique_id = s.unique_id
    GROUP BY t.unique_id
    """)
    cap_detail.createOrReplaceTempView("cap_stats")
    capped_count = cap_detail.agg({"capped_count": "sum"}).collect()[0][0] or 0

    spark.sql(f"""
    CREATE OR REPLACE TABLE {train_table} AS
    SELECT t.unique_id, t.ds,
      CASE
        WHEN t.y < s.q1 - {iqr_multiplier} * s.iqr THEN s.q1 - {iqr_multiplier} * s.iqr
        WHEN t.y > s.q3 + {iqr_multiplier} * s.iqr THEN s.q3 + {iqr_multiplier} * s.iqr
        ELSE t.y
      END AS y
    FROM {train_table} t
    JOIN (
      SELECT unique_id, PERCENTILE(y, 0.25) AS q1, PERCENTILE(y, 0.75) AS q3,
             PERCENTILE(y, 0.75) - PERCENTILE(y, 0.25) AS iqr
      FROM {train_table} GROUP BY unique_id
    ) s ON t.unique_id = s.unique_id
    """)
    print(f"Capped {int(capped_count):,} values using {iqr_multiplier}x IQR bounds")
else:
    spark.sql("CREATE OR REPLACE TEMP VIEW cap_stats AS SELECT CAST(NULL AS STRING) AS unique_id, 0 AS capped_count WHERE 1=0")
    print("Anomaly capping skipped (iqr_multiplier = 0)")

In [ ]:
train_table = f"{catalog}.{schema}.{use_case}_train_data"
report_table = f"{catalog}.{schema}.{use_case}_cleaning_report"

spark.sql(f"""
CREATE OR REPLACE TABLE {report_table} AS
SELECT
  t.unique_id,
  t.final_count,
  COALESCE(m.missing_count, 0) AS missing_filled,
  '{imputation_method}' AS imputation_method,
  COALESCE(c.capped_count, 0) AS anomalies_capped,
  CAST({iqr_multiplier} AS DOUBLE) AS iqr_multiplier,
  FALSE AS excluded,
  CAST(NULL AS STRING) AS exclusion_reason
FROM (SELECT unique_id, COUNT(*) AS final_count FROM {train_table} GROUP BY unique_id) t
LEFT JOIN missing_assessment m ON t.unique_id = m.unique_id
LEFT JOIN cap_stats c ON t.unique_id = c.unique_id
""")

n_series = spark.sql(f"SELECT COUNT(DISTINCT unique_id) AS n FROM {train_table}").collect()[0]["n"]
total_rows = spark.sql(f"SELECT COUNT(*) AS cnt FROM {train_table}").collect()[0]["cnt"]
date_range = spark.sql(f"SELECT MIN(ds) AS min_ds, MAX(ds) AS max_ds FROM {train_table}").collect()[0]

print(f"Data preparation complete for '{use_case}'")
print(f"  Training table: {train_table}")
print(f"  Cleaning report: {report_table}")
print(f"  Series: {n_series:,}")
print(f"  Rows: {total_rows:,}")
print(f"  Date range: {date_range['min_ds']} to {date_range['max_ds']}")
print(f"  Frequency: {freq}")
print(f"  Imputation: {imputation_method}")
print(f"  Anomaly capping: {'skipped' if iqr_multiplier == 0 else f'{iqr_multiplier}x IQR'}")